# TrafficVision — Entrenamiento YOLO11n
**Tesis:** Detección y lectura de placas vehiculares ecuatorianas  
**Modelo:** YOLO11n (nano) — optimizado para Colab Free T4  
**Dataset:** 7,576 imágenes combinadas (global + Ecuador)

---
### 📋 Orden de ejecución
| Celda | Descripción | Obligatoria |
|-------|-------------|-------------|
| 0 | Anti-desconexión | ✅ Siempre |
| 1 | Verificar GPU | ✅ Siempre |
| 2 | Instalar dependencias | ✅ Siempre |
| 3 | Montar Drive | ✅ Siempre |
| 4 | Verificar datasets | ✅ Siempre |
| 5 | Crear YAML | ✅ Siempre |
| 6 | **Entrenar** (nuevo) | 🔵 Primera vez |
| 7 | **Reanudar** (interrumpido) | 🟡 Si se cortó |
| 8 | Evaluar métricas | ✅ Al finalizar |
| 9 | Exportar modelo | ✅ Al finalizar |

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELDA 0 — Anti-desconexión + monitor de sesión
# ══════════════════════════════════════════════════════════════════
import time, threading

def heartbeat():
    """Evita la desconexión automática de Colab cada 90 min."""
    clicks = 0
    while True:
        time.sleep(45)
        clicks += 1
        try:
            from google.colab import output
            output.eval_js('document.querySelector("#top-toolbar").click()')
        except Exception:
            pass

t = threading.Thread(target=heartbeat, daemon=True)
t.start()

SESSION_START = time.time()
print('Anti-desconexión activo')
print('Si se interrumpe, usa CELDA 7 (Reanudar) — no pierdas el progreso.')

Anti-desconexión activo
Si se interrumpe, usa CELDA 7 (Reanudar) — no pierdas el progreso.


In [ ]:
# CELDA 1 — Verificar GPU y RAM disponible
!nvidia-smi

import torch, psutil, os

# GPU
cuda_ok = torch.cuda.is_available()
print(f'\n🔧 CUDA disponible: {cuda_ok}')
if cuda_ok:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'   GPU:  {gpu_name}')
    print(f'   VRAM: {gpu_mem:.1f} GB')
    # Recomendación de batch según VRAM
    if gpu_mem >= 14:
        rec_batch = 16
    elif gpu_mem >= 8:
        rec_batch = 8
    else:
        rec_batch = 4
    print(f'Batch recomendado: {rec_batch} (para imgsz=640)')
else:
    print(' Sin GPU — el entrenamiento será muy lento en CPU.')
    print(' Solución: Runtime → Cambiar tipo de entorno de ejecución → T4 GPU')

# RAM del sistema
ram = psutil.virtual_memory()
print(f'\n RAM sistema: {ram.available/1024**3:.1f} GB disponibles / {ram.total/1024**3:.1f} GB total')

# Espacio en disco
disk = psutil.disk_usage('/')
print(f' Disco /tmp:   {disk.free/1024**3:.1f} GB libres')

Sun Apr 26 01:12:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# CELDA 2 — Instalar dependencias
# ultralytics >= 8.3 incluye soporte completo para YOLO11
!pip install ultralytics -q

from ultralytics import YOLO
import ultralytics
print(f'✅ Ultralytics {ultralytics.__version__} instalado (soporta YOLO11)')

# Verificar versión mínima
major, minor = map(int, ultralytics.__version__.split('.')[:2])
if major < 8 or (major == 8 and minor < 3):
    print('⚠️  Versión antigua — puede no soportar YOLO11. Reinicia el runtime.')
else:
    print(f'✅ Versión compatible con YOLO11')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 49.8 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Ultralytics 8.4.41 instalado (soporta YOLO11)
✅ Versión compatible con YOLO11


In [ ]:
# CELDA 3 — Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Rutas base del proyecto
DRIVE_BASE  = '/content/drive/MyDrive/TrafficVision/datasets'
DRIVE_RUNS  = '/content/drive/MyDrive/TrafficVision/runs'
RUN_NAME    = 'yolo11n_combined_all'

# Crear carpeta de runs si no existe
import os
os.makedirs(DRIVE_RUNS, exist_ok=True)

print('✅ Google Drive montado')
print(f'   Datasets: {DRIVE_BASE}')
print(f'   Runs:     {DRIVE_RUNS}')

Mounted at /content/drive
✅ Google Drive montado
   Datasets: /content/drive/MyDrive/TrafficVision/datasets
   Runs:     /content/drive/MyDrive/TrafficVision/runs


In [ ]:
# CELDA 4 — Verificar datasets y estimar tiempo de entrenamiento
import os

datasets = {
    'license-plates (global)':  f'{DRIVE_BASE}/license-plates',
    'license-plates-ec-1':      f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-1',
    'license-plates-ec-2':      f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-2',
    'license-plates-ec-4':      f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-4',
}

total_train = 0
total_val   = 0
all_ok      = True

print('  VERIFICACIÓN DE DATASETS')

for name, path in datasets.items():
    exists = os.path.exists(path)
    if exists:
        train_path = f'{path}/train/images'
        val_path   = f'{path}/valid/images'
        n_train = len(os.listdir(train_path)) if os.path.exists(train_path) else 0
        n_val   = len(os.listdir(val_path))   if os.path.exists(val_path)   else 0
        total_train += n_train
        total_val   += n_val
        print(f'  ✅ {name}')
        print(f'     train: {n_train:,} imgs  |  val: {n_val:,} imgs')
    else:
        all_ok = False
        print(f'  ❌ {name} — NO ENCONTRADO')
        print(f'     Ruta esperada: {path}')

print(f'  TOTAL train: {total_train:,} imágenes')
print(f'  TOTAL val:   {total_val:,} imágenes')

# Estimación de tiempo (T4, batch=16, imgsz=640)
# ~2.5 seg/epoch por cada 1000 imgs en T4
secs_per_epoch = (total_train / 1000) * 2.5
total_mins     = (secs_per_epoch * 100) / 60
print(f'\n  Estimación para 100 épocas en T4 (batch=16):')
print(f'   ~{secs_per_epoch:.0f} seg/época  →  ~{total_mins:.0f} min totales ({total_mins/60:.1f} h)')
print(f'   Colab Free: máx ~4-5h por sesión.')

if total_mins > 240:
    safe_epochs = int((240 * 60) / secs_per_epoch)
    print(f'   Con este dataset, una sesión alcanza ~{safe_epochs} épocas.')
    print(f'   Usa save_period=5 y reanuda con CELDA 7 en la siguiente sesión.')

if not all_ok:
    print('\n Algunos datasets faltan. Verifica que estén en Drive antes de entrenar.')

  VERIFICACIÓN DE DATASETS
  ✅ license-plates (global)
     train: 7,057 imgs  |  val: 2,048 imgs
  ✅ license-plates-ec-1
     train: 54 imgs  |  val: 42 imgs
  ✅ license-plates-ec-2
     train: 90 imgs  |  val: 5 imgs
  ✅ license-plates-ec-4
     train: 375 imgs  |  val: 34 imgs
  TOTAL train: 7,576 imágenes
  TOTAL val:   2,129 imágenes

  Estimación para 100 épocas en T4 (batch=16):
   ~19 seg/época  →  ~32 min totales (0.5 h)
   Colab Free: máx ~4-5h por sesión.


In [ ]:
# CELDA 5 — Crear data_combined_all.yaml
import yaml, os

data = {
    'train': [
        f'{DRIVE_BASE}/license-plates/train/images',
        f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-1/train/images',
        f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-2/train/images',
        f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-4/train/images',
    ],
    # Validación solo con el dataset global (más imágenes = métricas más confiables)
    'val':  f'{DRIVE_BASE}/license-plates/valid/images',
    'test': f'{DRIVE_BASE}/license-plates/test/images',
    'nc':   1,
    'names': ['license plate'],
}

YAML_PATH = '/content/data_combined_all.yaml'
with open(YAML_PATH, 'w') as f:
    yaml.dump(data, f, default_flow_style=False, allow_unicode=True)

print('✅ data_combined_all.yaml creado')
print(f'   Ruta: {YAML_PATH}')
print(f'   Clases: {data["nc"]} ({data["names"]})')
print(f'   Carpetas train: {len(data["train"])}')
for p in data['train']:
    n = len(os.listdir(p)) if os.path.exists(p) else '❌ no existe'
    label = '/'.join(p.split('/')[-4:-2])
    print(f'     {label}: {n} imgs')

# Verificar labels también
print('\n  Verificando labels (.txt)...')
for p in data['train']:
    lp = p.replace('/images', '/labels')
    if os.path.exists(lp):
        n = len([f for f in os.listdir(lp) if f.endswith('.txt')])
        label = '/'.join(p.split('/')[-4:-2])
        print(f'     ✅ {label}: {n} labels')
    else:
        print(f'     Labels no encontrados en {lp}')

✅ data_combined_all.yaml creado
   Ruta: /content/data_combined_all.yaml
   Clases: 1 (['license plate'])
   Carpetas train: 4
     datasets/license-plates: 7057 imgs
     license-plates-ec-combined/license-plates-ec-1: 54 imgs
     license-plates-ec-combined/license-plates-ec-2: 90 imgs
     license-plates-ec-combined/license-plates-ec-4: 375 imgs

  Verificando labels (.txt)...
     ✅ datasets/license-plates: 7057 labels
     ✅ license-plates-ec-combined/license-plates-ec-1: 54 labels
     ✅ license-plates-ec-combined/license-plates-ec-2: 90 labels
     ✅ license-plates-ec-combined/license-plates-ec-4: 375 labels


In [ ]:
# CELDA 6 — ENTRENAR YOLO11n (primera vez)
# ⚠️  Solo ejecutar si NO existe un checkpoint previo.
#     Si el entrenamiento se interrumpió → usa CELDA 7 (Reanudar).
import os, time
from ultralytics import YOLO

# Verificar que no haya checkpoint previo accidentalmente
checkpoint = f'{DRIVE_RUNS}/{RUN_NAME}/weights/last.pt'
if os.path.exists(checkpoint):
    size_mb = os.path.getsize(checkpoint) / 1024**2
    print(f'⚠️  Ya existe un checkpoint: {checkpoint} ({size_mb:.1f} MB)')
    print('   Si quieres REANUDAR → usa CELDA 7')
    print('   Si quieres empezar DE CERO → cambia RUN_NAME arriba o borra la carpeta')
    print('   Deteniendo para no sobreescribir...')
    raise SystemExit('Checkpoint existente — usa CELDA 7 para reanudar.')

print('🚀 Iniciando entrenamiento YOLO11n...')
print(f'   Dataset:  {YAML_PATH}')
print(f'   Destino:  {DRIVE_RUNS}/{RUN_NAME}')
print()

model = YOLO('yolo11n.pt')

# ─── Parámetros optimizados
# batch=16:       Balance seguro VRAM/velocidad en T4 (14.9 GB)
# imgsz=640:      Estándar YOLO, no reducir (perdería precisión en placas pequeñas)
# cache='disk':   Evita recargar imágenes de Drive en cada época (crítico para Drive)
# save_period=5:  Guarda checkpoint cada 5 épocas → recuperación granular
# workers=2:      Drive es lento; más workers no ayudan y gastan RAM
# patience=20:    Early stopping más permisivo (dataset mixto puede tener ruido)
# cos_lr=True:    Convergencia más suave, mejor para datasets pequeños mezclados
# amp=True:       Mixed precision → 30% más rápido, menos VRAM

results = model.train(
    data          = YAML_PATH,
    epochs        = 100,
    imgsz         = 640,
    batch         = 16,          # ← seguro para T4; si sale OOM bajar a 8
    name          = RUN_NAME,
    project       = DRIVE_RUNS,
    patience      = 20,
    save          = True,
    save_period   = 5,           # ← checkpoint cada 5 épocas
    plots         = True,
    device        = 0,
    amp           = True,
    cos_lr        = True,
    cache         = 'disk',      # ← evita re-leer desde Drive en cada época
    workers       = 2,           # ← Drive es el cuello de botella, no la CPU
    warmup_epochs = 3,
    resume        = False,
    verbose       = True,
)

elapsed = (time.time() - SESSION_START) / 60
print(f'\n Entrenamiento completado en {elapsed:.1f} min')
print(f'   Modelo guardado en: {DRIVE_RUNS}/{RUN_NAME}/weights/best.pt')

🚀 Iniciando entrenamiento YOLO11n...
   Dataset:  /content/data_combined_all.yaml
   Destino:  /content/drive/MyDrive/TrafficVision/runs/yolo11n_combined_all

Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=disk, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/data_combined_all.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n

In [ ]:
# CELDA 7B — ENTRENAR solo datasets Ecuador (primera vez)
import yaml, os, time
from ultralytics import YOLO

RUN_NAME_EC = 'best_ecuador_yolo11'

# Verificar que no haya checkpoint previo
checkpoint_ec = f'{DRIVE_RUNS}/{RUN_NAME_EC}/weights/last.pt'
if os.path.exists(checkpoint_ec):
    size_mb = os.path.getsize(checkpoint_ec) / 1024**2
    print(f'⚠️  Ya existe un checkpoint: {checkpoint_ec} ({size_mb:.1f} MB)')
    print('   Si quieres REANUDAR → usa CELDA 7C')
    print('   Si quieres empezar DE CERO → borra la carpeta manualmente en Drive')
    raise SystemExit('Checkpoint existente — usa CELDA 7C para reanudar.')

# ── Crear YAML solo Ecuador
data_ec = {
    'train': [
        f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-1/train/images',
        f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-2/train/images',
        f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-4/train/images',
    ],
    # ec-1 tiene val/test propios; ec-2 y ec-4 no siempre tienen val
    'val':  f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-1/valid/images',
    'test': f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-1/test/images',
    'nc':   1,
    'names': ['license plate'],
}

YAML_EC = '/content/data_ecuador.yaml'
with open(YAML_EC, 'w') as f:
    yaml.dump(data_ec, f, default_flow_style=False, allow_unicode=True)

# Verificar datasets Ecuador
print('📂 Verificando datasets Ecuador...')
total_ec = 0
for p in data_ec['train']:
    if os.path.exists(p):
        n = len(os.listdir(p))
        total_ec += n
        label = p.split('/')[-3]
        print(f'   ✅ {label}: {n} imgs')
    else:
        print(f'   ❌ NO ENCONTRADO: {p}')

print(f'   Total train Ecuador: {total_ec} imágenes')

# Estimación de tiempo (T4, ~519 imgs, batch=16)
secs_ec = (total_ec / 1000) * 2.5
mins_ec = (secs_ec * 100) / 60
print(f'   ⏱️  Estimación 100 épocas: ~{mins_ec:.0f} min (~{mins_ec/60:.1f} h) — cabe en 1 sesión T4')

print(f'\n🚀 Iniciando entrenamiento {RUN_NAME_EC}...')
print(f'   Dataset: {YAML_EC}')
print(f'   Destino: {DRIVE_RUNS}/{RUN_NAME_EC}nuevo')
print()

model_ec = YOLO('yolo11n.pt')

results_ec = model_ec.train(
    data          = YAML_EC,
    epochs        = 100,
    imgsz         = 640,
    batch         = 16,
    name          = RUN_NAME_EC,
    project       = DRIVE_RUNS,
    patience      = 20,
    save          = True,
    save_period   = 5,
    plots         = True,
    device        = 0,
    amp           = True,
    cos_lr        = True,
    cache         = 'disk',
    workers       = 2,
    warmup_epochs = 3,
    resume        = False,
    verbose       = True,
)

elapsed = (time.time() - SESSION_START) / 60
print(f'\n Entrenamiento Ecuador completado en {elapsed:.1f} min')
print(f'   Modelo: {DRIVE_RUNS}/{RUN_NAME_EC}/weights/best.pt')
print(f'\n Comparativa disponible en CELDA 8 (evalúa ambos modelos)')

📂 Verificando datasets Ecuador...
   ✅ license-plates-ec-1: 54 imgs
   ✅ license-plates-ec-2: 90 imgs
   ✅ license-plates-ec-4: 375 imgs
   Total train Ecuador: 519 imágenes
   ⏱️  Estimación 100 épocas: ~2 min (~0.0 h) — cabe en 1 sesión T4

🚀 Iniciando entrenamiento best_ecuador_yolo11...
   Dataset: /content/data_ecuador.yaml
   Destino: /content/drive/MyDrive/TrafficVision/runs/best_ecuador_yolo11nuevo

Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=disk, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/data_ecuador.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, fo

In [ ]:
# CELDA 7 — REANUDAR entrenamiento interrumpido
# Usar cuando se desconectó o se agotó el tiempo de GPU.
# Ejecuta celdas 0-5 antes!
#
# OPTIMIZACIÓN: copia el dataset a SSD local antes de entrenar.
# El scanning desde Drive tarda ~80 min; desde SSD local <30 seg.

import glob, os, shutil, time, yaml
from ultralytics import YOLO

# ── Rutas ─────────────────────────────────────────────────────────────────────
DRIVE_DATASET_GLOBAL = f'{DRIVE_BASE}/license-plates'
DRIVE_DATASET_EC     = f'{DRIVE_BASE}/license-plates-ec-combined'
LOCAL_BASE           = '/content/datasets'
LOCAL_GLOBAL         = f'{LOCAL_BASE}/license-plates'
LOCAL_EC             = f'{LOCAL_BASE}/license-plates-ec-combined'
LOCAL_YAML           = '/content/data_combined_all_local.yaml'

# ── 1. Copiar datasets a SSD local ────────────────────────────────────────────
# shutil.copytree es compatible con el filesystem FUSE de Drive (rsync no lo es)
os.makedirs(LOCAL_BASE, exist_ok=True)

for src, dst, label in [
    (DRIVE_DATASET_GLOBAL, LOCAL_GLOBAL, 'license-plates (global)'),
    (DRIVE_DATASET_EC,     LOCAL_EC,     'license-plates-ec-combined'),
]:
    if os.path.exists(dst):
        print(f'✅ {label} ya está en local — omitiendo copia')
    else:
        print(f'📦 Copiando {label}... (puede tardar 3-8 min la primera vez)')
        t0 = time.time()
        shutil.copytree(src, dst)
        mins = (time.time() - t0) / 60
        n = sum(len(files) for _, _, files in os.walk(dst))
        print(f'   ✅ {n:,} archivos copiados en {mins:.1f} min  →  {dst}')

# ── 2. Crear YAML apuntando a /content/ ───────────────────────────────────────
with open(YAML_PATH, 'r') as f:   # YAML_PATH definido en CELDA 5
    cfg = yaml.safe_load(f)

def local_path(p):
    return p.replace(DRIVE_BASE, LOCAL_BASE) if isinstance(p, str) else p

cfg['train'] = [local_path(p) for p in cfg['train']] if isinstance(cfg.get('train'), list) else local_path(cfg.get('train'))
cfg['val']   = local_path(cfg.get('val', ''))
cfg['test']  = local_path(cfg.get('test', ''))

with open(LOCAL_YAML, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False, allow_unicode=True)
print(f'\n📄 YAML local creado: {LOCAL_YAML}')

# ── 3. Buscar checkpoint (tolerando sufijos -2/-3/-4) ─────────────────────────
def encontrar_last_pt(drive_runs, run_name):
    """Retorna la ruta a last.pt tolerando sufijos numéricos en el nombre del run."""
    exacto = f'{drive_runs}/{run_name}/weights/last.pt'
    if os.path.exists(exacto):
        return exacto, run_name

    variantes = sorted(
        glob.glob(f'{drive_runs}/{run_name}-*/weights/last.pt'),
        reverse=True
    )
    if variantes:
        run_real      = variantes[0].split('/weights/')[0]
        run_name_real = os.path.basename(run_real)
        print(f'ℹ️  Run con sufijo detectado: {run_name_real}')
        print(f'   (YOLO creó el sufijo porque la carpeta ya existía)')
        return variantes[0], run_name_real

    cualquier = sorted(
        glob.glob(f'{drive_runs}/{run_name}*/weights/*.pt'),
        key=os.path.getmtime, reverse=True
    )
    if cualquier:
        run_real      = cualquier[0].split('/weights/')[0]
        run_name_real = os.path.basename(run_real)
        print(f'ℹ️  last.pt no encontrado, usando más reciente: {cualquier[0]}')
        return cualquier[0], run_name_real

    return None, None

last_pt, run_name_real = encontrar_last_pt(DRIVE_RUNS, RUN_NAME)

if last_pt is None:
    print(f'❌ No se encontró ningún checkpoint para "{RUN_NAME}" en:')
    print(f'   {DRIVE_RUNS}/')
    print()
    print('   Verifica que Drive esté montado y que el entrenamiento')
    print('   haya llegado al menos al primer save_period (época 5).')
    print('   Si no hay checkpoint → ejecuta CELDA 6 para comenzar de cero.')
    raise FileNotFoundError('Sin checkpoint para reanudar.')

size_mb = os.path.getsize(last_pt) / 1024**2
print(f'\n✅ Checkpoint: {last_pt}  ({size_mb:.1f} MB)')

try:
    import torch
    ckpt  = torch.load(last_pt, map_location='cpu', weights_only=False)
    epoch = ckpt.get('epoch', '?')
    print(f'   Última época guardada: {epoch}/100')
    del ckpt
except Exception:
    print('   (No se pudo leer la época del checkpoint)')

# ── 4. Reanudar entrenamiento ──────────────────────────────────────────────────
print('\n🔁 Reanudando entrenamiento...')

model   = YOLO(last_pt)
results = model.train(
    data        = LOCAL_YAML,      # ← SSD local, no Drive
    epochs      = 100,
    imgsz       = 640,
    batch       = 16,
    name        = run_name_real,
    project     = DRIVE_RUNS,      # ← checkpoints siguen guardándose en Drive
    exist_ok    = True,
    patience    = 20,
    save        = True,
    save_period = 5,
    plots       = True,
    device      = 0,
    amp         = True,
    cos_lr      = True,
    cache       = 'ram',           # ← RAM más rápida cuando imágenes están en SSD
    workers     = 4,               # ← sin cuello de botella Drive, más workers ayudan
    resume      = True,
    verbose     = True,
)

print('\n✅ Entrenamiento reanudado y completado')


✅ license-plates (global) ya está en local — omitiendo copia
📦 Copiando license-plates-ec-combined... (puede tardar 3-8 min la primera vez)
   ✅ 1,367 archivos copiados en 9.4 min  →  /content/datasets/license-plates-ec-combined

📄 YAML local creado: /content/data_combined_all_local.yaml
ℹ️  Run con sufijo detectado: yolo11n_combined_all-4
   (YOLO creó el sufijo porque la carpeta ya existía)

✅ Checkpoint: /content/drive/MyDrive/TrafficVision/runs/yolo11n_combined_all-4/weights/last.pt  (15.2 MB)
   Última época guardada: 14/100

🔁 Reanudando entrenamiento...
Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/data_combined_all.yaml, degrees=0.0, determinist

In [ ]:
# CELDA 7C — REANUDAR entrenamiento Ecuador interrumpido
import yaml, glob, os
from ultralytics import YOLO

RUN_NAME_EC = 'yolo11n_ecuador_combined'

# Recrear YAML Ecuador si el runtime se reinició
YAML_EC = '/content/data_ecuador.yaml'
if not os.path.exists(YAML_EC):
    data_ec = {
        'train': [
            f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-1/train/images',
            f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-2/train/images',
            f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-4/train/images',
        ],
        'val':  f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-1/valid/images',
        'test': f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-1/test/images',
        'nc':   1,
        'names': ['license plate'],
    }
    with open(YAML_EC, 'w') as f:
        yaml.dump(data_ec, f, default_flow_style=False, allow_unicode=True)
    print(f'✅ YAML Ecuador recreado: {YAML_EC}')
else:
    print(f'✅ YAML Ecuador ya existe: {YAML_EC}')

# Buscar checkpoint Ecuador
last_pt_ec = f'{DRIVE_RUNS}/{RUN_NAME_EC}/weights/last.pt'
if not os.path.exists(last_pt_ec):
    candidates = glob.glob(f'{DRIVE_RUNS}/{RUN_NAME_EC}/weights/*.pt')
    if candidates:
        last_pt_ec = sorted(candidates)[-1]
        print(f'ℹ️  last.pt no encontrado, usando: {last_pt_ec}')
    else:
        print(f'❌ No hay checkpoint en {DRIVE_RUNS}/{RUN_NAME_EC}/')
        raise FileNotFoundError('Sin checkpoint Ecuador para reanudar.')

size_mb = os.path.getsize(last_pt_ec) / 1024**2
print(f'✅ Checkpoint encontrado: {last_pt_ec} ({size_mb:.1f} MB)')

try:
    import torch
    ckpt  = torch.load(last_pt_ec, map_location='cpu', weights_only=False)
    epoch = ckpt.get('epoch', '?')
    print(f'   Última época guardada: {epoch}/100')
    del ckpt
except Exception:
    print('   (No se pudo leer la época del checkpoint)')

print('\n🔁 Reanudando entrenamiento Ecuador...')

model_ec = YOLO(last_pt_ec)
results_ec = model_ec.train(
    data        = YAML_EC,
    epochs      = 100,
    imgsz       = 640,
    batch       = 16,
    name        = RUN_NAME_EC,
    project     = DRIVE_RUNS,
    exist_ok    = True,
    patience    = 20,
    save        = True,
    save_period = 5,
    plots       = True,
    device      = 0,
    amp         = True,
    cos_lr      = True,
    cache       = 'disk',
    workers     = 2,
    resume      = True,
    verbose     = True,
)

print('\n✅ Entrenamiento Ecuador reanudado y completado')

✅ YAML Ecuador recreado: /content/data_ecuador.yaml
✅ Checkpoint encontrado: /content/drive/MyDrive/TrafficVision/runs/yolo11n_ecuador_combined/weights/last.pt (20.3 MB)
   Última época guardada: 18/100

🔁 Reanudando entrenamiento Ecuador...
Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=disk, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/data_ecuador.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0

In [ ]:
# CELDA 8 — EVALUAR métricas (combined_all vs Ecuador)
# ══════════════════════════════════════════════════════════════════
# FIX: busca el run real aunque YOLO le haya puesto sufijo (-2/-3/-4)
# FIX: recrea YAML_EC y YAML_PATH si el runtime se reinició
# FIX: autodetecta el nombre del run Ecuador sin importar el que usaste
# ══════════════════════════════════════════════════════════════════
import glob, os, yaml
from ultralytics import YOLO

# ── Recrear YAMLs si el runtime se reinició ───────────────────────
YAML_PATH = '/content/data_combined_all.yaml'
if not os.path.exists(YAML_PATH):
    data_all = {
        'train': [
            f'{DRIVE_BASE}/license-plates/train/images',
            f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-1/train/images',
            f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-2/train/images',
            f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-4/train/images',
        ],
        'val':  f'{DRIVE_BASE}/license-plates/valid/images',
        'test': f'{DRIVE_BASE}/license-plates/test/images',
        'nc': 1, 'names': ['license plate'],
    }
    with open(YAML_PATH, 'w') as f:
        yaml.dump(data_all, f, default_flow_style=False, allow_unicode=True)
    print(f'♻️  YAML combined_all recreado')

YAML_EC = '/content/data_ecuador.yaml'
if not os.path.exists(YAML_EC):
    data_ec = {
        'train': [
            f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-1/train/images',
            f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-2/train/images',
            f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-4/train/images',
        ],
        'val':  f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-1/valid/images',
        'test': f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-1/test/images',
        'nc': 1, 'names': ['license plate'],
    }
    with open(YAML_EC, 'w') as f:
        yaml.dump(data_ec, f, default_flow_style=False, allow_unicode=True)
    print(f'♻️  YAML Ecuador recreado')

# ── Función: busca run aunque tenga sufijo (-2/-3/-4) ─────────────
def encontrar_best_pt(drive_runs, run_name):
    """Devuelve best.pt del run, tolerando sufijos que YOLO agrega."""
    exacto = f'{drive_runs}/{run_name}/weights/best.pt'
    if os.path.exists(exacto):
        return exacto, run_name
    # Buscar con sufijo numérico, ordenar por más reciente
    variantes = sorted(glob.glob(f'{drive_runs}/{run_name}-*/weights/best.pt'), reverse=True)
    if variantes:
        run_real = '/'.join(variantes[0].split('/')[:-2])
        return variantes[0], os.path.basename(run_real)
    return None, None

def evaluar_modelo(best_pt, yaml_path, label):
    if not best_pt or not os.path.exists(best_pt):
        print(f'   ⚠️  {label}: no encontrado')
        return None
    size_mb = os.path.getsize(best_pt) / 1024**2
    print(f'\n✅ Evaluando {label}: {best_pt} ({size_mb:.1f} MB)')
    model   = YOLO(best_pt)
    metrics = model.val(
        data=yaml_path, imgsz=640, device=0,
        batch=16, plots=True, save_json=False,
    )
    return metrics

# ── Evaluar combined_all ───────────────────────────────────────────
best_combined, run_combined = encontrar_best_pt(DRIVE_RUNS, 'yolo11n_combined_all')
if run_combined and run_combined != 'yolo11n_combined_all':
    print(f'ℹ️  Run combined encontrado con sufijo: {run_combined}')
m_combined = evaluar_modelo(best_combined, YAML_PATH, f'combined_all [{run_combined}]')

# ── Evaluar Ecuador — autodetectar nombre usado en 7B ─────────────
ec_nombres = ['yolo11n_ecuador_combined', 'best_ecuador_yolo11']
best_ec, run_ec = None, None
for nombre in ec_nombres:
    best_ec, run_ec = encontrar_best_pt(DRIVE_RUNS, nombre)
    if best_ec:
        break
# Búsqueda genérica si no coincide ninguno
if not best_ec:
    for pt in glob.glob(f'{DRIVE_RUNS}/*/weights/best.pt'):
        rn = pt.split('/')[-3]
        if 'combined_all' not in rn and 'ecuador' in rn.lower():
            best_ec, run_ec = pt, rn
            break

if run_ec:
    print(f'\n🔍 Run Ecuador detectado: {run_ec}')
m_ecuador = evaluar_modelo(best_ec, YAML_EC, f'Ecuador [{run_ec}]') if best_ec else None
if not best_ec:
    print('\nℹ️  No se encontró modelo Ecuador — mostrando solo combined_all.')

# ── Tabla comparativa ─────────────────────────────────────────────
print('\n')
print('╔══════════════════════════════════════════════════════════════╗')
print('║         COMPARATIVA DE MODELOS — PLACAS ECUATORIANAS          ║')
print('╠═══════════════════════╦══════════════════╦════════════════════╣')
print('║ Métrica               ║  combined_all    ║  solo Ecuador      ║')
print('╠═══════════════════════╬══════════════════╬════════════════════╣')

def fmt(val): return f'{val:.4f} ({val*100:.1f}%)' if val is not None else 'N/A'

rows = [
    ('mAP@50',    m_combined.box.map50 if m_combined else None, m_ecuador.box.map50 if m_ecuador else None),
    ('mAP@50-95', m_combined.box.map   if m_combined else None, m_ecuador.box.map   if m_ecuador else None),
    ('Precisión', m_combined.box.mp    if m_combined else None, m_ecuador.box.mp    if m_ecuador else None),
    ('Recall',    m_combined.box.mr    if m_combined else None, m_ecuador.box.mr    if m_ecuador else None),
]
for nombre, val_c, val_e in rows:
    print(f'║ {nombre:<21s} ║ {fmt(val_c):<16s} ║ {fmt(val_e):<18s} ║')
print('╚═══════════════════════╩══════════════════╩════════════════════╝')

# ── Conclusión automática ─────────────────────────────────────────
if m_combined and m_ecuador:
    diff = (m_combined.box.map50 - m_ecuador.box.map50) * 100
    if diff > 2:
        print(f'\n📈 combined_all supera a solo-Ecuador en {diff:+.1f} pp mAP@50')
        print('   → El dataset global mejora la generalización del modelo.')
    elif diff < -2:
        print(f'\n📈 solo-Ecuador supera a combined_all en {-diff:+.1f} pp mAP@50')
        print('   → El modelo especializado rinde mejor en placas locales.')
    else:
        print(f'\n🟰 Rendimiento similar ({diff:+.1f} pp) — combined_all preferible')
        print('   por mejor generalización con más datos.')
elif m_combined:
    map50 = m_combined.box.map50
    delta = (map50 - 0.974) * 100
    print(f'\n{"📈" if delta >= 0 else "📉"} vs YOLOv8n anterior (97.4%): {delta:+.1f} pp')
    print('  ✅ Excelente' if map50 >= 0.95 else ('  ⚠️  Aceptable' if map50 >= 0.85 else '  ❌ Bajo'))

♻️  YAML Ecuador recreado
ℹ️  Run combined encontrado con sufijo: yolo11n_combined_all-4

✅ Evaluando combined_all [yolo11n_combined_all-4]: /content/drive/MyDrive/TrafficVision/runs/yolo11n_combined_all-4/weights/best.pt (15.2 MB)
Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.6±0.2 ms, read: 0.1±0.0 MB/s, size: 25.1 KB)
val: Scanning /content/drive/MyDrive/TrafficVision/datasets/license-plates/valid/labels.cache... 2048 images, 3 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2048/2048 373.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 128/128 4.8s/it 10:20
                   all       2048       2195      0.982      0.923      0.962      0.657
Speed: 0.5ms preprocess, 3.6ms inference, 0.0ms loss, 0.8ms postprocess per image
Results saved to /content/runs/detect/val-

In [ ]:
# CELDA 9 — Exportar modelo para producción
# Genera una copia en /content/ (RAM local) para descarga inmediata
# Y deja best.pt en Drive para uso en el backend.
import shutil, os
from ultralytics import YOLO

best_pt   = f'{DRIVE_RUNS}/{RUN_NAME}/weights/best.pt'
export_pt = f'/content/yolo11n_trafficvision_best.pt'

if not os.path.exists(best_pt):
    print(f'❌ No se encontró {best_pt}')
else:
    # Copiar a /content para descarga rápida
    shutil.copy2(best_pt, export_pt)
    size_mb = os.path.getsize(export_pt) / 1024**2
    print(f'✅ Modelo copiado a: {export_pt} ({size_mb:.1f} MB)')

    # Descarga directa desde Colab
    from google.colab import files
    print('\n📥 Iniciando descarga del modelo...')
    files.download(export_pt)

    print('\n📁 Ruta permanente en Drive:')
    print(f'   {best_pt}')
    print('\n🔧 Para usar en el backend (plate_detector.py):')
    print('   MODEL_PATH = "ml/models/trained/yolo11n_combined_all/best.pt"')

    # Resumen final del run
    results_csv = f'{DRIVE_RUNS}/{RUN_NAME}/results.csv'
    if os.path.exists(results_csv):
        import pandas as pd
        df = pd.read_csv(results_csv)
        df.columns = df.columns.str.strip()
        best_row = df.loc[df['metrics/mAP50(B)'].idxmax()]
        best_epoch = int(best_row['epoch']) + 1
        best_map50 = best_row['metrics/mAP50(B)']
        print(f'\n📊 Mejor época: {best_epoch}/100  →  mAP@50 = {best_map50:.4f} ({best_map50*100:.1f}%)')

---
## 💡 Guía rápida — Colab Free

### Si se desconecta durante el entrenamiento
1. Abre el notebook de nuevo
2. Ejecuta celdas **0 → 5** (anti-disco, GPU, instalar, Drive, verificar, YAML)
3. Ejecuta **CELDA 7** (Reanudar) — YOLO retoma desde el último `save_period`

### Señales de que va bien
- `box_loss` y `cls_loss` bajando cada época ✅
- `mAP50` subiendo progresivamente ✅  
- GPU mem ~4-6 GB (con batch=16) ✅

### Si sale `CUDA out of memory`
- Bajar `batch` de 16 → 8 en celdas 6 y 7
- Si persiste: `batch=4, imgsz=416`

### Tiempos estimados (T4, batch=16)
| Épocas | Tiempo estimado |
|--------|----------------|
| 30     | ~55 min        |
| 50     | ~92 min        |
| 100    | ~3 h           |

### Advertencia sobre segmentos
El WARNING `len(segments) != len(boxes)` en el dataset ec-1 es inofensivo —  
YOLO ignora las anotaciones de segmentación y usa solo los bounding boxes.
